In [9]:
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
# from itertools import cycle, product
from pathlib import Path

%cd /g/data/w42/dr6273/work/wind_drought/
import src.wind as wind
import src.utils as utils

/g/data/w42/dr6273/work/wind_drought


In [2]:
%load_ext autoreload
%autoreload 2

### Wind drought (WDD) data

In [3]:
def load_wind(dataset, variable, region, scenario=None, averaging="equal"):
    """ Return da of wind speed data for given criteria """
    wind_path = "/g/data/ng72/dr6273/work/projects/wind_drought/data/"+dataset+"/power_grid/"
    directory_path = Path(wind_path)

    # Filter by variable
    matching_files = [f.name for f in directory_path.iterdir() if f.is_file() and variable in f.name]

    # Filter by region
    matching_files = [f for f in matching_files if region in f]

    # Filter by scenario
    if scenario is not None:
        matching_files = [f for f in matching_files if scenario in f]

    # Filter by averaging method
    matching_files = [f for f in matching_files if averaging in f]
    
    if len(matching_files) > 1:
        raise ValueError("Need to filter more files")
        
    return xr.open_dataset(wind_path + matching_files[0])

- ERA5 and BARRA-R2
- capacity factor
- MJJ 10th percentile drought-day threshold
- weighted SC30 scenario for NEM/SE region

In [7]:
wdd_dict = {}
for dataset in ["ERA5", "BARRA-R2"]:
    wdd_dict[dataset] = {}

    for variable in ["capacity_factor"]:
        wdd_dict[dataset][variable] = {}

        for region in ["NEM", "SE", "SWIS", "NWIS"]:
            
            if region in ["NEM", "SE"]:
                for scenario in ["SC30"]:
                    wdd_dict[dataset][variable][region+"-"+scenario] = {}

                    for averaging in ["weighted"]:
                        da = load_wind(dataset, variable, region, scenario, averaging)[variable]
                        pctl = utils.sel_month(da, [5, 6, 7]).quantile(0.1, "time")
                        wdd = wind.calc_wdd_timeseries(da, pctl)
                        
                        wdd_dict[dataset][variable][region+"-"+scenario][averaging] = wdd

            else:
                da = load_wind(dataset, variable, region)[variable]
                pctl = utils.sel_month(da, [5, 6, 7]).quantile(0.1, "time")
                wdd = wind.calc_wdd_timeseries(da, pctl)
                
                wdd_dict[dataset][variable][region] = wdd

#### Longest droughts

In [43]:
def get_droughts(da):
    """Return dataframe of drought duration, start date and end date"""
    years = da.time.dt.year.values
    df = pd.DataFrame(index=years)
    df['WDD'] = da.values.astype('int32')
    df['start_date'] = pd.to_datetime(da.time.dt.strftime('%Y-%m-%d'))
    df['end_date'] = pd.to_datetime(da.time.dt.strftime('%Y-%m-%d')) + pd.to_timedelta(df['WDD'] - 1, unit='D')
    return df.set_index('start_date')

In [44]:
events_dict = utils.map_leaves(
    wdd_dict,
    lambda value: get_droughts(value)
)

In [54]:
df = events_dict['BARRA-R2']['capacity_factor']['NEM-SC30']['weighted']
df

,WDD,end_date
start_date,,
1979-03-08,1,1979-03-08
1979-04-06,1,1979-04-06
1979-05-08,2,1979-05-09
1979-05-17,2,1979-05-18
1979-07-18,1,1979-07-18
...,...,...
2025-05-13,4,2025-05-16
2025-06-11,2,2025-06-12
2025-06-26,4,2025-06-29


In [95]:
def split_droughts(df, wdd_col="WDD", q1=0.95, q2=0.99):
    q_low, q_high = df[wdd_col].quantile([q1, q2])
    print("n events:", len(df))
    print("low WDD thresh:", q_low, "high WDD thresh:", q_high)

    top = df[df[wdd_col] >= q_high].sort_values(wdd_col, ascending=False)
    other = df[
        (df[wdd_col] >= q_low) & (df[wdd_col] < q_high)
    ].sort_values(wdd_col, ascending=False)

    print("n top events:", len(top), "n other events:", len(other))
    print()

    return {"top": top, "other": other}

In [96]:
subset_dict = utils.map_leaves(events_dict, split_droughts)

n events: 685
low WDD thresh: 3.0 high WDD thresh: 5.0
n top events: 9 n other events: 45

n events: 696
low WDD thresh: 3.0 high WDD thresh: 4.0499999999999545
n top events: 7 n other events: 42

n events: 586
low WDD thresh: 3.0 high WDD thresh: 4.0
n top events: 8 n other events: 26

n events: 621
low WDD thresh: 3.0 high WDD thresh: 6.0
n top events: 9 n other events: 79

n events: 516
low WDD thresh: 3.0 high WDD thresh: 5.0
n top events: 9 n other events: 40

n events: 553
low WDD thresh: 3.0 high WDD thresh: 5.0
n top events: 8 n other events: 39

n events: 515
low WDD thresh: 3.0 high WDD thresh: 3.0
n top events: 33 n other events: 0

n events: 634
low WDD thresh: 4.0 high WDD thresh: 6.0
n top events: 10 n other events: 23



### Synoptic Weather Types

In [10]:
# Australian Synoptic Weather Type data from forked repository
swt = pd.read_csv(
    "../Australian-synoptic-weather-types/SWT_climatology/SWT_climatology.csv",
    index_col=0,
    parse_dates=True
)

In [11]:
ordered_SWTs = [
    "WH-A", "WH-B", "WH-C", "WH-D",
    "CH-A", "CH-B",
    "EH-A", "EH-B", "EH-C", "EH-D", "EH-E",
    "TH-A", "TH-B", "TH-C",
    "FH-A", "FH-B", "FH-C",
    "WCT-A", "WCT-B",
    "COL-A", "COL-B", "COL-C", "COL-D", "COL-E", "COL-F",
    "AM-A", "AM-B", "AM-C", "AM-D", "AM-E"
]

In [12]:
missing_values = set(ordered_SWTs) - set(swt["assigned_SWT"].dropna().unique())

if missing_values:
    print("Missing from swt:", sorted(missing_values))
else:
    print("All ordered_SWTs are present in swt.")

All ordered_SWTs are present in swt.


In [13]:
# Plotting bars for each SWT category in the groups
swt_colors=[
    (134/255,0/255,34/255),(241/255,0/255,241/255),(255/255,134/255,255/255),(255/255,241/255,255/255), #Pink WH
    (255/255,204/255,51/255),(255/255,245/255,204/255), #Yellow CH
    (153/255,15/255,15/255),(178/255,44/255,44/255),(204/255,81/255,81/255),(229/255,126/255,126/255),(255/255,178/255,178/255), #Red EH
    (153/255,84/255,15/255),(204/255,142/255,81/255),(255/255,216/255, 178/255), #Brown TH
    (107/255,153/255,15/255),(163/255,204/255,81/255),(195/255,229/255,126/255), #Green FH (133/255,178/255,44/255),
    (66/255,44/255,178/255),(143/255,126/255,229/255), #Purple WCT
    (5/255,67/255,113/255),(15/255,107/255,153/255),(44/255,133/255,178/255),(81/255,163/255,204/255),(126/255,195/255,229/255),(178/255,229/255,255/255), #Blue COL
    (0/255,60/255,48/255),(1/255,102/255,95/255),(53/255,151/255,143/255),(128/255,205/255,193/255),(199/255,234/255,229/255)
]

In [14]:
colour_by_swt = dict(zip(ordered_SWTs, swt_colors))

Get SWT occurrences during droughts

In [25]:
for a, b in zip(droughts_subset['nem-2030']['top'].index, droughts_subset['nem-2030']['top']['end_date']):
    print(a,b)

1997-05-16 00:00:00 1997-05-24 00:00:00
2002-05-22 00:00:00 2002-05-29 00:00:00
2010-05-14 00:00:00 2010-05-21 00:00:00
2005-05-14 00:00:00 2005-05-19 00:00:00
1992-05-14 00:00:00 1992-05-18 00:00:00
1995-05-17 00:00:00 1995-05-21 00:00:00
2000-07-06 00:00:00 2000-07-10 00:00:00
2001-05-01 00:00:00 2001-05-05 00:00:00
2006-07-23 00:00:00 2006-07-27 00:00:00


In [4]:
wind.calc_wdd_timeseries?

Signature: wind.calc_wdd_timeseries(timeseries, pctl)
Docstring: Calculate wind drought durations (WDD) for a single time series
File:      /g/data/w42/dr6273/work/wind_drought/src/wind.py
Type:      function